# CRNN Baseline — Colab GPU Training

Trains the Devanagari CRNN baseline on a free Colab T4.

**Before running:** `Runtime → Change runtime type → T4 GPU`.

Run cells top to bottom. See `docs/GPU_TRAINING.md` for details.

In [1]:
# 1. Confirm GPU is on
import torch
assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> T4 GPU"
print(torch.cuda.get_device_name(0))

Tesla T4


In [2]:
# 2. Get the code (branch: ml) and install deps
!git clone -b ml https://github.com/Sanskriti-poudel/Devnagari_Handwriting_Recognition.git
%cd Devnagari_Handwriting_Recognition
!pip install -q opencv-python numpy pillow tqdm scikit-learn

Cloning into 'Devnagari_Handwriting_Recognition'...
remote: Enumerating objects: 65, done.
remote: Counting objects: 100% (65/65), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 65 (delta 9), reused 61 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (65/65), 1.14 MiB | 15.96 MiB/s, done.
Resolving deltas: 100% (9/9), done.
/content/Devnagari_Handwriting_Recognition


In [6]:
# 3. Mount Drive and point the data loader at the preprocessed dataset
from google.colab import drive
drive.mount('/content/drive')

import os
os.environ["DEVNAGARI_DATA_ROOT"] = "/content/drive/MyDrive/Devnagari Datasets/Preprocessed"

# Sanity check: the folder should contain train/ and test/
print(os.listdir(os.environ["DEVNAGARI_DATA_ROOT"]))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['test', 'train']


In [ ]:
# 4. (Optional but recommended on first run) Scan for corrupt/unreadable images.
# Drive-mounted reads occasionally fail; this lists any bad files up front.
!python Preprocessing/verify_dataset.py --out corrupt_images.txt || true

[verify] DATA_ROOT = /content/drive/MyDrive/Devnagari Datasets/Preprocessed
[verify] scanning split 'train': 73609 images
  4% 2944/73609 [21:32<7:02:42,  2.79it/s]

In [ ]:
# 5. Train (auto-detects CUDA; batch_size defaults to 64 on GPU,
#    override with CRNN_BATCH_SIZE if you hit out-of-memory)
!python models/crnn/train.py

In [ ]:
# 6. Save outputs back to Drive — Colab wipes local files on disconnect!
!mkdir -p "/content/drive/MyDrive/Devnagari Datasets/artifacts/crnn"
!cp models/crnn/checkpoints/best_model.pth "/content/drive/MyDrive/Devnagari Datasets/artifacts/crnn/"
!cp logs/crnn_training.csv "/content/drive/MyDrive/Devnagari Datasets/artifacts/crnn/"
print("Saved best_model.pth + crnn_training.csv to Drive.")

## After training

- Download `crnn_training.csv` for the report's training curves (Phase 3).
- Keep `best_model.pth` on Drive — weights are git-ignored; share via Drive (see Contract A in `plans/README.md`).
- Next: evaluation on the held-out test set (Accuracy / CER / WER).